# Advanced Credit Card Fraud Detection
Detect fraudulent credit card transactions from PCA-transformed features.

## Project Overview
Binary classification on a highly imbalanced fraud detection dataset. Focus on handling class imbalance with calibrated thresholds and class weights.

## Learning Objectives
- Handle extreme class imbalance (0.17% fraud)
- Use PR-AUC and recall as primary metrics
- Apply class weights and threshold tuning
- Compare boosting models on fraud data

## Problem Statement
Given PCA-transformed transaction features, detect whether a transaction is fraudulent (Class=1) or legitimate (Class=0).

## Why This Project Matters
Credit card fraud costs billions annually. Even small improvements in detection save millions. False negatives (missed fraud) are far more costly than false positives.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: creditcard.csv |
| **Target** | Class (0=legit, 1=fraud) |
| **Rows** | ~284,807 |
| **Features** | V1-V28 (PCA), Time, Amount |
| **Imbalance** | 99.83% legit, 0.17% fraud |

## Dataset Source & License
Kaggle: mlg-ulb/creditcardfraud. ULB Machine Learning Group. Open license for research.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
from sklearn.metrics import precision_recall_curve, average_precision_score
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [ ]:
TARGET = 'Class'
MAX_ROWS = 50000

## Dataset Loading

In [ ]:
csv_path = os.path.join(SAVE_DIR, 'creditcard.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}')
df = pd.read_csv(csv_path)
print(f'Full shape: {df.shape}')

# Stratified sample to keep fraud proportion
if len(df) > MAX_ROWS:
    fraud = df[df[TARGET] == 1]
    legit = df[df[TARGET] == 0].sample(MAX_ROWS - len(fraud), random_state=SEED)
    df = pd.concat([fraud, legit]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Sampled shape: {df.shape}')
df.head()

## Data Validation

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nClass distribution:')
print(df[TARGET].value_counts())
print(f'Fraud rate: {df[TARGET].mean():.4%}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Class Distribution')

axes[0,1].hist(df['Amount'].clip(upper=500), bins=50, edgecolor='black')
axes[0,1].set_title('Transaction Amount (clipped)')

for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    subset = df[df[TARGET] == label]
    axes[1,0].hist(subset['Amount'].clip(upper=500), bins=50, alpha=0.5, color=color, label=f'Class {label}')
axes[1,0].legend()
axes[1,0].set_title('Amount by Class')

corr_with_target = df.corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
corr_with_target.head(10).plot.bar(ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Top Correlations with Fraud')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [ ]:
print(df[TARGET].value_counts())
print(f'\nFraud rate: {df[TARGET].mean():.4%}')
print('This is extremely imbalanced — accuracy is meaningless, use PR-AUC and Recall.')

## Train/Test Split

In [ ]:
# Scale Amount and drop Time
from sklearn.preprocessing import StandardScaler
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df = df.drop(columns=['Time', 'Amount'])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4%}')

## Preprocessing
Scaled Amount feature. Dropped Time (not useful for fraud detection). PCA features already standardized.

## Feature Engineering
Features are PCA-transformed — minimal engineering needed. Scaled Amount for consistency.

## Baseline Model

In [ ]:
bl = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
bl_proba = bl.predict_proba(X_test)[:, 1]
print(f'Baseline LR (balanced): Recall={recall_score(y_test, bl_pred):.4f}  Precision={precision_score(y_test, bl_pred):.4f}')
print(f'  PR-AUC={average_precision_score(y_test, bl_proba):.4f}  ROC-AUC={roc_auc_score(y_test, bl_proba):.4f}')

## LazyPredict Benchmark

In [ ]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

## FLAML AutoML

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
models = {}
# CatBoost with auto class weights
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0, auto_class_weights='Balanced')
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM with class weight
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, is_unbalance=True)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost with scale_pos_weight
scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss', scale_pos_weight=scale)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    print(f"{name}: Recall={recall_score(y_test, p):.4f}  Precision={precision_score(y_test, p):.4f}  PR-AUC={average_precision_score(y_test, proba):.4f}")

## Top 3 Model Selection

In [ ]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    all_results[name] = {
        'Recall': recall_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'F1': f1_score(y_test, p),
        'PR-AUC': average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
    }
results_df = pd.DataFrame(all_results).T.sort_values('PR-AUC', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3 (by PR-AUC): {top3_names}')

## Final Evaluation of Top 3

In [ ]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    final_results[name] = {
        'Recall': recall_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'F1': f1_score(y_test, p),
        'PR-AUC': average_precision_score(y_test, proba),
    }
    print(f"{name}:")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Recall', 'PR-AUC']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

## Error Analysis

In [ ]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

# PR Curve
prec, rec, thresholds = precision_recall_curve(y_test, proba)
axes[1].plot(rec, prec)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

# Score distribution
axes[2].hist(proba[y_test == 0], bins=50, alpha=0.5, label='Legit', density=True)
axes[2].hist(proba[y_test == 1], bins=50, alpha=0.5, label='Fraud', density=True)
axes[2].legend()
axes[2].set_title('Score Distribution')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

fn = ((preds == 0) & (y_test == 1)).sum()
fp = ((preds == 1) & (y_test == 0)).sum()
print(f'False negatives (missed fraud): {fn}')
print(f'False positives (false alarms): {fp}')

## Interpretation & Business Insight
V14, V17, V12 are the most important PCA features for fraud. High recall is critical — missing a fraud costs much more than a false alarm.

## Limitations
- PCA features are uninterpretable
- No temporal patterns captured
- Extreme imbalance challenges all models

## How to Improve
- Threshold tuning for optimal recall/precision tradeoff
- Anomaly detection approaches (Isolation Forest, Autoencoders)
- Sequential transaction patterns

## Production Considerations
- Real-time scoring with sub-100ms latency
- Adaptive thresholds per merchant/region
- Human review queue for borderline cases

## Common Mistakes
- Using accuracy as metric (99.8% by predicting all legit)
- Not using class weights
- Random oversampling without careful evaluation

## Mini Challenge
1. Tune the threshold to maximize F1
2. Try Isolation Forest as anomaly detector
3. Compare SMOTE vs class weights

## Final Summary
Built fraud detection models on imbalanced data. PR-AUC and recall are the right metrics — accuracy is misleading.

In [ ]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))